In [1]:
!pip install -q sktime

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/diss-datasets'

import os
os.makedirs(DATA_DIR, exist_ok=True)
print('Currently in DATA_DIR:', os.listdir(DATA_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Currently in DATA_DIR: ['MP8.npy', 'Rowing.npy', 'MP50.npy', 'CMJ_text_embeddings_train.npy', 'CMJ_text_embeddings_test.npy', 'CMJ_text_results.txt', 'MP8_text_embeddings_train.npy', 'MP8_text_embeddings_test.npy', 'MP8_text_results.txt', 'MP50_text_embeddings_train.npy', 'MP50_text_embeddings_test.npy', 'MP50_text_results.txt', 'SYNTH_text_embeddings_train.npy', 'SYNTH_text_embeddings_test.npy', 'SYNTH_text_results.txt', 'Rowing_text_embeddings_train.npy', 'Rowing_text_embeddings_test.npy', 'Rowing_text_results.txt', 'chbmit_parts', 'CHBMIT_crosspatient.npy', 'Chronos2_CHBMIT_train_emb.npy', 'Chronos2_CHBMIT_test_emb.npy', 'Chronos2_CHBMIT_perm_train_emb.npy', 'Chronos2_CHBMIT_perm_test_emb.npy', 'Chronos2_CHBMIT_desync_train_emb.npy', 'Chronos2_CHBMIT_desync_test_emb.npy', 'MantisV2_CHBMIT_train_emb.npy', 'Chronos2_CHBMIT_chanpool_train_emb.npy', 'Chronos2_

In [3]:
from google.colab import files
import zipfile

uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATA_DIR)

print('Now in DATA_DIR:', os.listdir(DATA_DIR))

Saving drive-download-20260612T140644Z-3-001 2.zip to drive-download-20260612T140644Z-3-001 2 (1).zip
Now in DATA_DIR: ['MP8.npy', 'Rowing.npy', 'MP50.npy', 'CMJ_text_embeddings_train.npy', 'CMJ_text_embeddings_test.npy', 'CMJ_text_results.txt', 'MP8_text_embeddings_train.npy', 'MP8_text_embeddings_test.npy', 'MP8_text_results.txt', 'MP50_text_embeddings_train.npy', 'MP50_text_embeddings_test.npy', 'MP50_text_results.txt', 'SYNTH_text_embeddings_train.npy', 'SYNTH_text_embeddings_test.npy', 'SYNTH_text_results.txt', 'Rowing_text_embeddings_train.npy', 'Rowing_text_embeddings_test.npy', 'Rowing_text_results.txt', 'chbmit_parts', 'CHBMIT_crosspatient.npy', 'Chronos2_CHBMIT_train_emb.npy', 'Chronos2_CHBMIT_test_emb.npy', 'Chronos2_CHBMIT_perm_train_emb.npy', 'Chronos2_CHBMIT_perm_test_emb.npy', 'Chronos2_CHBMIT_desync_train_emb.npy', 'Chronos2_CHBMIT_desync_test_emb.npy', 'MantisV2_CHBMIT_train_emb.npy', 'Chronos2_CHBMIT_chanpool_train_emb.npy', 'Chronos2_CHBMIT_chanpool_test_emb.npy', 'f

In [4]:
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

DATASETS = {
    'CMJ':     {'file': 'CMJ.npy',                 'binary': False},
    'MP8':     {'file': 'MP8.npy',                 'binary': False},
    'MP50':    {'file': 'MP50.npy',                'binary': False},
    'Rowing':  {'file': 'Rowing.npy',               'binary': False},
    'SYNTH':   {'file': 'SYNTH.npy',                'binary': True},
    'CHB-MIT': {'file': 'CHBMIT_crosspatient.npy',  'binary': True},
}

for name, meta in DATASETS.items():
    path = f'{DATA_DIR}/{meta["file"]}'
    print(name, 'found' if os.path.exists(path) else 'MISSING', path)

CMJ found /content/drive/MyDrive/diss-datasets/CMJ.npy
MP8 found /content/drive/MyDrive/diss-datasets/MP8.npy
MP50 found /content/drive/MyDrive/diss-datasets/MP50.npy
Rowing found /content/drive/MyDrive/diss-datasets/Rowing.npy
SYNTH found /content/drive/MyDrive/diss-datasets/SYNTH.npy
CHB-MIT found /content/drive/MyDrive/diss-datasets/CHBMIT_crosspatient.npy


In [5]:
def load_dataset(fname):
    d = np.load(f'{DATA_DIR}/{fname}', allow_pickle=True).item()
    X_tr = d['train']['X'].astype(np.float64)
    y_tr = np.array([int(v) for v in d['train']['y']])
    X_te = d['test']['X'].astype(np.float64)
    y_te = np.array([int(v) for v in d['test']['y']])
    return X_tr, y_tr, X_te, y_te

def evaluate(y_true, y_pred, binary):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    out = {'accuracy': acc, 'macro_f1': macro_f1}
    if binary:
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1)
        out.update({'pos_precision': p, 'pos_recall': r, 'pos_f1': f1})
    return out

In [6]:
from sktime.transformations.panel.rocket import MultiRocketMultivariate, MultiRocket

def run_multirocket_multivariate(X_tr, y_tr, X_te, y_te, binary):
    t0 = time.time()
    rocket = MultiRocketMultivariate()
    rocket.fit(X_tr)
    feat_tr = rocket.transform(X_tr).astype(np.float32)
    feat_te = rocket.transform(X_te).astype(np.float32)
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(feat_tr, y_tr)
    pred = clf.predict(feat_te)
    elapsed = time.time() - t0
    result = evaluate(y_te, pred, binary)
    result['seconds'] = elapsed
    del feat_tr, feat_te
    return result

def run_multirocket_singlechannel(X_tr, y_tr, X_te, y_te, binary, num_kernels=None):
    t0 = time.time()
    feats_tr, feats_te = [], []
    n_channels = X_tr.shape[1]
    kwargs = {'num_kernels': num_kernels} if num_kernels else {}
    for ch in range(n_channels):
        r = MultiRocket(**kwargs)
        Xc_tr = X_tr[:, ch:ch+1, :]
        Xc_te = X_te[:, ch:ch+1, :]
        r.fit(Xc_tr)
        feats_tr.append(r.transform(Xc_tr).astype(np.float32))
        feats_te.append(r.transform(Xc_te).astype(np.float32))
    feat_tr = pd.concat(feats_tr, axis=1)
    feat_te = pd.concat(feats_te, axis=1)
    del feats_tr, feats_te
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(feat_tr, y_tr)
    pred = clf.predict(feat_te)
    elapsed = time.time() - t0
    result = evaluate(y_te, pred, binary)
    result['seconds'] = elapsed
    result['num_kernels'] = num_kernels if num_kernels else 'default'
    del feat_tr, feat_te
    return result

In [7]:
from sktime.transformations.panel.rocket import MiniRocketMultivariate, MiniRocket

def run_minirocket_multivariate(X_tr, y_tr, X_te, y_te, binary):
    t0 = time.time()
    rocket = MiniRocketMultivariate()
    rocket.fit(X_tr)
    feat_tr = rocket.transform(X_tr)
    feat_te = rocket.transform(X_te)
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(feat_tr, y_tr)
    pred = clf.predict(feat_te)
    elapsed = time.time() - t0
    result = evaluate(y_te, pred, binary)
    result['seconds'] = elapsed
    return result

def run_minirocket_singlechannel(X_tr, y_tr, X_te, y_te, binary):
    t0 = time.time()
    feats_tr, feats_te = [], []
    n_channels = X_tr.shape[1]
    for ch in range(n_channels):
        r = MiniRocket()
        Xc_tr = X_tr[:, ch:ch+1, :]
        Xc_te = X_te[:, ch:ch+1, :]
        r.fit(Xc_tr)
        feats_tr.append(r.transform(Xc_tr))
        feats_te.append(r.transform(Xc_te))
    feat_tr = pd.concat(feats_tr, axis=1)
    feat_te = pd.concat(feats_te, axis=1)
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(feat_tr, y_tr)
    pred = clf.predict(feat_te)
    elapsed = time.time() - t0
    result = evaluate(y_te, pred, binary)
    result['seconds'] = elapsed
    return result

In [8]:
import gc

RESULTS_PATH = f'{DATA_DIR}/rocket_all_results.csv'

if os.path.exists(RESULTS_PATH):
    done_df = pd.read_csv(RESULTS_PATH)
    done_keys = set(zip(done_df['dataset'], done_df['method'], done_df['variant']))
    print(f'Resuming, {len(done_keys)} results already saved')
else:
    done_df = pd.DataFrame()
    done_keys = set()

def save_result(r):
    global done_df
    row = pd.DataFrame([r])
    done_df = pd.concat([done_df, row], ignore_index=True)
    done_df.to_csv(RESULTS_PATH, index=False)

RISK_THRESHOLD = 1_000_000

def risk_metric(X_tr):
    return X_tr.shape[0] * X_tr.shape[2]

for name, meta in DATASETS.items():
    print(f'\n=== {name} ===')
    X_tr, y_tr, X_te, y_te = load_dataset(meta['file'])
    n_channels = X_tr.shape[1]
    risk = risk_metric(X_tr)
    print(f'shape train {X_tr.shape}  test {X_te.shape}  channels {n_channels}  risk {risk}')

    key = (name, 'MiniRocket', 'multivariate')
    if key not in done_keys:
        r = run_minirocket_multivariate(X_tr, y_tr, X_te, y_te, meta['binary'])
        r.update({'dataset': name, 'method': 'MiniRocket', 'variant': 'multivariate'})
        print(f"MiniRocket multivariate   acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} ({r['seconds']:.1f}s)")
        save_result(r)
    else:
        print('MiniRocket multivariate   skipped, already done')

    key = (name, 'MiniRocket', 'single-channel')
    if key not in done_keys:
        r = run_minirocket_singlechannel(X_tr, y_tr, X_te, y_te, meta['binary'])
        r.update({'dataset': name, 'method': 'MiniRocket', 'variant': 'single-channel'})
        print(f"MiniRocket single-channel acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} ({r['seconds']:.1f}s)")
        save_result(r)
    else:
        print('MiniRocket single-channel skipped, already done')

    key = (name, 'MultiRocket', 'multivariate')
    if key not in done_keys:
        r = run_multirocket_multivariate(X_tr, y_tr, X_te, y_te, meta['binary'])
        r.update({'dataset': name, 'method': 'MultiRocket', 'variant': 'multivariate'})
        print(f"MultiRocket multivariate  acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} ({r['seconds']:.1f}s)")
        save_result(r)
    else:
        print('MultiRocket multivariate  skipped, already done')

    key = (name, 'MultiRocket', 'single-channel')
    if key not in done_keys:
        if risk >= RISK_THRESHOLD:
            print(f'  SKIPPED: risk {risk} >= {RISK_THRESHOLD}, MultiRocket single-channel omitted for this dataset (memory constraint on available RAM)')
            r = {
                'accuracy': np.nan, 'macro_f1': np.nan,
                'dataset': name, 'method': 'MultiRocket', 'variant': 'single-channel',
                'seconds': 0, 'num_kernels': 'skipped: memory constraint'
            }
            save_result(r)
        else:
            r = run_multirocket_singlechannel(X_tr, y_tr, X_te, y_te, meta['binary'])
            r.update({'dataset': name, 'method': 'MultiRocket', 'variant': 'single-channel'})
            print(f"MultiRocket single-channel acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} ({r['seconds']:.1f}s)")
            save_result(r)
    else:
        print('MultiRocket single-channel skipped, already done')

    del X_tr, y_tr, X_te, y_te
    gc.collect()

Resuming, 24 results already saved

=== CMJ ===
shape train (419, 3, 384)  test (179, 3, 384)  channels 3  risk 160896
MiniRocket multivariate   skipped, already done
MiniRocket single-channel skipped, already done
MultiRocket multivariate  skipped, already done
MultiRocket single-channel skipped, already done

=== MP8 ===
shape train (1426, 8, 161)  test (595, 8, 161)  channels 8  risk 229586
MiniRocket multivariate   skipped, already done
MiniRocket single-channel skipped, already done
MultiRocket multivariate  skipped, already done
MultiRocket single-channel skipped, already done

=== MP50 ===
shape train (1426, 50, 161)  test (595, 50, 161)  channels 50  risk 229586
MiniRocket multivariate   skipped, already done
MiniRocket single-channel skipped, already done
MultiRocket multivariate  skipped, already done
MultiRocket single-channel skipped, already done

=== Rowing ===
shape train (1773, 45, 161)  test (760, 45, 161)  channels 45  risk 285453
MiniRocket multivariate   skipped, al

In [ ]:
REDUCED_KERNELS = 1000
results = []

for name, meta in DATASETS.items():
    print(f'\n=== {name} ===')
    X_tr, y_tr, X_te, y_te = load_dataset(meta['file'])
    n_channels = X_tr.shape[1]
    print(f'shape train {X_tr.shape}  test {X_te.shape}  channels {n_channels}')

    r = run_minirocket_multivariate(X_tr, y_tr, X_te, y_te, meta['binary'])
    r.update({'dataset': name, 'method': 'MiniRocket', 'variant': 'multivariate'})
    print(f"MiniRocket multivariate   acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} ({r['seconds']:.1f}s)")
    results.append(r)

    r = run_minirocket_singlechannel(X_tr, y_tr, X_te, y_te, meta['binary'])
    r.update({'dataset': name, 'method': 'MiniRocket', 'variant': 'single-channel'})
    print(f"MiniRocket single-channel acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} ({r['seconds']:.1f}s)")
    results.append(r)

    r = run_multirocket_multivariate(X_tr, y_tr, X_te, y_te, meta['binary'])
    r.update({'dataset': name, 'method': 'MultiRocket', 'variant': 'multivariate'})
    print(f"MultiRocket multivariate  acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} ({r['seconds']:.1f}s)")
    results.append(r)

    kernels = REDUCED_KERNELS if n_channels >= 30 else None
    if kernels:
        print(f'  (reducing MultiRocket to {kernels} kernels per channel for the single-channel loop, {n_channels} channels)')
    r = run_multirocket_singlechannel(X_tr, y_tr, X_te, y_te, meta['binary'], num_kernels=kernels)
    r.update({'dataset': name, 'method': 'MultiRocket', 'variant': 'single-channel'})
    print(f"MultiRocket single-channel acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} ({r['seconds']:.1f}s)")
    results.append(r)


=== CMJ ===
shape train (419, 3, 384)  test (179, 3, 384)  channels 3
MiniRocket multivariate   acc=0.9441 macroF1=0.9442 (3.0s)
MiniRocket single-channel acc=0.9497 macroF1=0.9497 (2.0s)
MultiRocket multivariate  acc=0.9497 macroF1=0.9500 (9.7s)
MultiRocket single-channel acc=0.9385 macroF1=0.9390 (21.2s)

=== MP8 ===
shape train (1426, 8, 161)  test (595, 8, 161)  channels 8
MiniRocket multivariate   acc=0.7378 macroF1=0.7412 (4.3s)
MiniRocket single-channel acc=0.6891 macroF1=0.6850 (9.5s)
MultiRocket multivariate  acc=0.6672 macroF1=0.6695 (16.7s)
MultiRocket single-channel acc=0.6790 macroF1=0.6742 (88.8s)

=== MP50 ===
shape train (1426, 50, 161)  test (595, 50, 161)  channels 50
MiniRocket multivariate   acc=0.7765 macroF1=0.7768 (6.3s)
MiniRocket single-channel acc=0.7580 macroF1=0.7585 (53.0s)
MultiRocket multivariate  acc=0.6034 macroF1=0.6026 (21.7s)
  (reducing MultiRocket to 1000 kernels per channel for the single-channel loop, 50 channels)
MultiRocket single-channel acc=

In [ ]:
df = pd.read_csv(RESULTS_PATH)
pd.set_option('display.width', 140)
print(df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

mini = df[df['method'] == 'MiniRocket']
pivot = mini.pivot(index='dataset', columns='variant', values='macro_f1')
pivot = pivot.reindex(['CMJ', 'MP8', 'MP50', 'Rowing', 'SYNTH', 'CHB-MIT'])

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(pivot))
width = 0.35
ax.bar(x - width/2, pivot['multivariate'], width, label='Multivariate (channel-aware)')
ax.bar(x + width/2, pivot['single-channel'], width, label='Single-channel (channel-independent)')
ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_ylabel('Macro F1')
ax.set_title('MiniRocket: channel-independence penalty across domains')
ax.legend()
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/rocket_channel_penalty.png', dpi=150)
plt.show()

In [ ]:
chbmit_scores = {
    'Majority baseline': 0.6667,
    'Correlation features': 0.7318,
    'Band power features': 0.8218,
    'Band power + correlation': 0.8678,
    'MantisV2 (frozen probe)': 0.7950,
    'Chronos-2 (frozen probe)': 0.8199,
}
chbmit_rocket = df[df['dataset'] == 'CHB-MIT'].set_index(['method', 'variant'])['accuracy']
for (method, variant), val in chbmit_rocket.items():
    chbmit_scores[f'{method} ({variant})'] = val

items = sorted(chbmit_scores.items(), key=lambda kv: kv[1])
labels, vals = zip(*items)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(labels, vals)
ax.set_xlabel('Accuracy')
ax.set_title('CHB-MIT: all methods compared')
ax.set_xlim(0.6, 0.9)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/chbmit_all_methods.png', dpi=150)
plt.show()

In [ ]:
import json
with open(f'{DATA_DIR}/rocket_summary.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)
print('saved')